In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 02b_rq1_qual_visualisations.py
# Purpose of Script: Visualise all synthetic media identified in Qualitative SOR data.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [210]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [147]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect("/content/working.duckdb")

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_clean = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [190]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Total SORS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_all_sors = pd.read_parquet(f"{path_samp}rq1_final.parquet")
df_all_sors = df_all_sors[["platform","date","total_sor_entries"]]
df_all_sors["date"] = pd.to_datetime(df_all_sors["date"])
df_all_sors["platform"] = df_all_sors["platform"].str.capitalize()

In [247]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Monthly User Numbers - Using Per Million
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_users = pd.read_csv(f"{path_out}00_average_monthly_users_clean.csv").reset_index(drop=True)
df_users = df_users[["platform","date","users"]]
df_users["date"] = pd.to_datetime(df_users["date"])
df_users["platform"] = df_users["platform"].str.capitalize()
df_users["users_million"] = df_users["users"]/1000000

In [195]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Synthetic Qualitative SORS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
### Tiktok
df_syn_tiktok = con.execute(f""" select date, count(*) as syn_sor_entries
                                 from read_parquet('{path_out}02_rq1_qual_14_tiktok_*.parquet')
                                 group by date""").df()

df_syn_tiktok["platform"] = "tiktok"

### Youtube
df_syn_youtube = con.execute(f""" select date, count(*) as syn_sor_entries
                                 from read_parquet('{path_out}02_rq1_qual_14_youtube_*.parquet')
                                 group by date""").df()

df_syn_youtube["platform"] = "youtube"

In [196]:
### Append
df_syn_sors = pd.concat([df_syn_tiktok,df_syn_youtube], ignore_index=False)
df_syn_sors["date"] = pd.to_datetime(df_syn_sors["date"])
df_syn_sors["platform"] = df_syn_sors["platform"].str.capitalize()

In [197]:
# Join All Tables
df = df_all_sors.merge(df_syn_sors, on=["platform","date"], how="left")
df = df.merge(df_users, on=["platform","date"], how="left")

In [198]:
# Fill NAs
df = df.fillna({"syn_sor_entries" : 0})

In [199]:
# Derived Columns - Relative to Total and User Numbers
df["syn_rel_to_total"] = np.where(df["total_sor_entries"] > 0,
    df["syn_sor_entries"] / df["total_sor_entries"],
    np.nan)
df = df.fillna({"syn_rel_to_total" : 0})
df["syn_rel_to_users"] = (df["syn_sor_entries"]/df["users"])*100
df["syn_rel_to_mil_users"] = (df["syn_sor_entries"]/df["users_million"])*100

# Derived Columns - 30 Day Rolling Averages
df["synthetic_ma30"] = (df.groupby("platform")["syn_sor_entries"].transform(lambda x: x.rolling(30, min_periods=1).mean()))

In [200]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Tabular Analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Summary Table
summary = (df.groupby("platform").agg(
          days=("date", "count"),
          total_sors=("total_sor_entries", "sum"),
          synthetic_sors=("syn_sor_entries", "sum"),
          mean_synthetic_day=("syn_sor_entries", "mean"),
          median_synthetic_day=("syn_sor_entries", "median"),
          mean_percent=("syn_rel_to_total", "mean"),
          max_percent=("syn_rel_to_total", "max"),
          mean_users=("users", "mean")).reset_index())

In [202]:
# Overall
overall = (df.groupby("platform")[["total_sor_entries", "syn_sor_entries"]]
      .sum()
      .reset_index())

overall["overall_percent"] = (overall["syn_sor_entries"] /overall["total_sor_entries"] * 100)

In [235]:
# Volatility Analysis
### Assess volatility per million users - Standard Deviation
vol = (df.groupby("platform")["syn_rel_to_mil_users"]
      .agg(mean="mean", median="median",
          sd="std", minimum="min", maximum="max")
      .reset_index())

### Calculate Coefficient of Variation
vol["cv"] = vol["sd"] / vol["mean"]
vol["cv_percent"] = vol["cv"] * 100

### Format
vol = vol.round({"mean": 2, "median": 2, "sd": 2,
                 "minimum": 2, "maximum": 2, "cv": 2,
                 "cv_percent": 1})

### Remove Platforms with 0 Synthetic Media
vol = vol[vol["cv"] > 0]

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Visualisations ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
### Nominal Synthetic Media Moderation
plt.figure(figsize=(12,6))

sns.lineplot(data=df, x="date", y="syn_sor_entries", hue="platform", marker="o")

plt.legend(title="Platform", fontsize=9)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.xlabel("Date", fontsize=9)
plt.tight_layout()
plt.ylabel("Synthetic SORs: Nominal Frequency [#]", fontsize=9)
plt.title("SORs Moderating Synthetic Media Nominal Frequency - Qualitative Analysis", fontsize=9)
plt.show()

# Export
plt.savefig(f"{path_out}02b_rq1_qual_1_synthetic_media_nominal_freq.pdf", bbox_inches="tight")

In [ ]:
### Relative Synthetic Media Moderation to Total SORs
plt.figure(figsize=(12,6))

sns.lineplot(data=df, x="date", y="syn_rel_to_total", hue="platform", marker="o")

plt.legend(title="Platform", fontsize=9)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.xlabel("Date", fontsize=9)
plt.tight_layout()
plt.ylabel("Synthetic SORs: Relative Frequency to Total Submitted SORs [%]", fontsize=9)
plt.title("SORs Moderating Synthetic Media Relative Frequency to Total Submitted SORs - Qualitative Analysis", fontsize=9)
plt.show()

# Export
plt.savefig(f"{path_out}02b_rq1_qual_2_synthetic_media_rel_to_total.pdf", bbox_inches="tight")

In [ ]:
### Relative Synthetic Media Moderation to Monthly Users per Million
plt.figure(figsize=(12,6))

sns.lineplot(data=df, x="date", y="syn_rel_to_mil_users", hue="platform",
             marker="o")

plt.legend(title="Platform", fontsize=9)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.legend(title="Platform", fontsize=9)
plt.xlabel("Date", fontsize=9)
plt.tight_layout()
plt.ylabel("Synthetic SORs: Relative Frequency to Average Monthly Users (per Million) [%]", fontsize=9)
plt.title("SORs Moderating Synthetic Media Relative Frequency to Average Monthly Users - Qualitative Analysis", fontsize=9)
plt.show()

# Export
plt.savefig(f"{path_out}02b_rq1_qual_3_synthetic_media_rel_to_users.pdf", bbox_inches="tight")

In [ ]:
### 30-Day Moving Average
plt.figure(figsize=(12,6))

sns.lineplot(data=df, x="date", y="synthetic_ma30", hue="platform",
             marker="o")

plt.legend(title="Platform", fontsize=9)
plt.legend(title="Platform", fontsize=9)
plt.xlabel("Date", fontsize=9)
plt.xticks(rotation=45, fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.ylabel("30-day average synthetic SORs", fontsize=9)
plt.title("Synthetic Media Moderation Over Time - 30 Day Average Nominal Frequency", fontsize=9)
plt.show()

# Export
plt.savefig(f"{path_out}02b_rq1_qual_4_synthetic_media_30_day_average.pdf", bbox_inches="tight")

In [245]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Tables ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Summary
summary.to_csv(f"{path_out}02b_rq1_qual_5_summary.csv")

# Overall
overall.to_csv(f"{path_out}02b_rq1_qual_6_overall.csv")

# Volatility
vol.to_csv(f"{path_out}02b_rq1_qual_7_vol.csv")